# Load the combined data from 10% of QUD and DRED Datasets

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path


data_path = "../FL Flower Pipeline/Dataset/SSFL_DATA/data_com_glob_train.csv"
test_data_path = "../FL Flower Pipeline/Dataset/data_com_glob_test.csv"

data = pd.read_csv(data_path)
test_data = pd.read_csv(test_data_path)

data_train_x = data.iloc[:, :-1].values
data_train_y = data.iloc[:, -1].values

data_test_x = test_data.iloc[:, :-1].values
data_test_y = test_data.iloc[:, -1].values

In [ ]:
if data_train_x.shape[0] != data_train_y.shape[0]:
    print("The length of input and ouput in training data doesn't match! Please check you data!!!!")
    """ 
    elif data_test_x.shape[0] != data_test_y.shape[0]:
    print("The length of input and ouput in test data doesn't match! Please check you data!!!!") 
    """
else:
    print("You input and output data lenghts matches. You are good to go!")

# Load & Train Model 

In [ ]:
import tensorflow as tf
from Utils.utils import build_model

input_shape = input_shape = (5, )
model_name = "test_model4"

model = build_model(input_shape,  model_name=model_name)

model.summary()

In [ ]:
history = model.fit(data_train_x, data_train_y, epochs=100, batch_size=32, validation_split=0.1)

In [ ]:
total_loss, total_accuracy = model.evaluate(
            data_test_x, data_test_y, batch_size=32, steps=10)

print(total_loss, total_accuracy)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, multilabel_confusion_matrix


y_pred = np.argmax(model.predict(data_test_x), axis=-1)

# Calculate metrics
accuracy = accuracy_score(data_test_y, y_pred)
precision = precision_score(data_test_y, y_pred, average='macro')
recall = recall_score(data_test_y, y_pred, average='macro')
f1_s = f1_score(data_test_y, y_pred, average='macro')

# Detailed classification report for sensitivity (recall for each class)
class_report = classification_report(data_test_y, y_pred)

conf_mat = confusion_matrix(data_test_y, y_pred)
cm_per_class = multilabel_confusion_matrix(data_test_y, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall (Sensitivity): {recall}")
print(f"F1-Score: {f1_s}")
print("\nClassification Report:")
print(class_report)
print(conf_mat)
#print(cm_per_class)

In [ ]:
eval_maat = []
classes = [0, 1, 2, 3, 4, "Overall"]

for i in range(5):
    TN = cm_per_class[i][0][0]
    FP = cm_per_class[i][0][1]
    FN = cm_per_class[i][1][0]
    TP = cm_per_class[i][1][1]

    Class = classes[i]
    Accuracy = round(100*(TP+TN)/(TP+TN+FP+FN), 2)
    Precision = round(100*(TP)/(TP+FP), 2)
    Recall = round(100*(TP)/(TP+FN), 2)
    F1_score = round((2*Precision*Recall)/(Precision+Recall), 2)
    Specipicity = round(100*(TN)/(TN+FP), 2)

    eval_maat.append([Class, Precision, Recall, Specipicity, F1_score, Accuracy])

headers = ["Class", "Precision", "Recall", "Specipicity", "F1_score", "Accuracy"]
temp_tab = pd.DataFrame(eval_maat, columns=headers)

s = np.sum(conf_mat, axis=1)

Class = classes[5] 
Accuracy = round(temp_tab["Accuracy"].dot(s)/np.sum(s), 2)
Precision = round(temp_tab["Precision"].dot(s)/np.sum(s), 2)
Recall = round(temp_tab["Recall"].dot(s)/np.sum(s), 2)
F1_score = round(temp_tab["F1_score"].dot(s)/np.sum(s), 2)
Specipicity = round(temp_tab["Specipicity"].dot(s)/np.sum(s), 2)
#Accuracy = round(temp_tab["Accuracy"].dot(s)/np.sum(s), 2)

eval_maat.append([Class, Precision, Recall, Specipicity, F1_score, Accuracy])

results = pd.DataFrame(eval_maat, columns=headers)

print(results)